# Lezione 8: La Pipeline di Addestramento Completa
**File**: [`train.py`](file:///g:/My%20Drive/PhD%20&%20Research/Temporal%20GNN/cfm_grn/train.py)  
**Ruolo nel progetto**: Orchestra la simulazione dei dati, l'accoppiamento Optimal Transport, la discesa del gradiente con regolarizzazione e la validazione sul gene held-out.


---
## 1. Il Ciclo di Addestramento del Conditional Flow Matching

Per ogni batch di cellule di controllo $x_0$ e perturbate $x_1$:
1. **Campionamento del Tempo**: $t \sim \mathcal{U}(0, 1)$.
2. **Interpolazione e Target**: $x_t = (1-t)x_0 + t x_1 + \sigma_{\min}\epsilon$ e $u_t = x_1 - x_0$.
3. **Predizione**: $v = v_\theta(x_t, t, c)$.
4. **Loss Totale**:
$$\mathcal{L} = \frac{1}{B} \|v - u_t\|^2 + \lambda_{\text{reg}} \cdot \text{Mean}(M_\theta)$$
5. **Ottimizzazione**: Passo dell'algoritmo `AdamW`.


---
## 2. Setup e Import


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
from torch.optim import AdamW

from cfm.flow_matcher import OptimalTransportCFM
from models.graph_vector_field import GraphVectorField
from data.synthetic_grn import SyntheticGRNSimulator
from data.dataset import create_dataloaders
from train import train_epoch, evaluate_model
print('Pipeline di training caricata!')


---
## 3. Mini-Esperimento di Addestramento Guidato

Addestriamo una rete per 10 epoche e tracciamo la curva di apprendimento in tempo reale.


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Addestramento su device: {device}')

# 1. Generazione dati
sim = SyntheticGRNSimulator(num_genes=12, density=0.25, seed=42)
benchmark_data = sim.generate_benchmark_dataset(cells_per_condition=250)
held_out_gene = benchmark_data['perturbed_genes'][0]

train_loader, _ = create_dataloaders(benchmark_data, batch_size=64, held_out_genes=[held_out_gene])

# 2. Inizializzazione modello e CFM
model = GraphVectorField(
    num_genes=12,
    edge_index=sim.edge_index.to(device),
    perturbation_dim=12,
    hidden_dim=32,
    num_layers=2,
).to(device)

cfm = OptimalTransportCFM(sigma_min=1e-3)
optimizer = AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)

# 3. Training loop
epochs = 10
history = []

print('Inizio training...')
for ep in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, cfm, optimizer, device)
    history.append(loss)
    print(f'Epoca {ep:02d}/{epochs:02d} - Loss CFM: {loss:.4f}')

print('Training completato!')


In [ ]:
# Plot della loss di addestramento
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), history, 'b-o', linewidth=2)
plt.title('Curva di Discesa della Loss CFM', fontsize=12, fontweight='bold')
plt.xlabel('Epoca', fontsize=11)
plt.ylabel('Loss L2 Regressione', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()
